# Validação da extração Azure Blob (agrupamento por dia) — análise completa

Este notebook roda uma **análise completa** do carregamento de interações via
`kpibench.data.azure_blob_loader`, após a correção do bug de parsing de data
(pasta `date=YYYYMMDD` dos blobs não era reconhecida — `_extract_blob_date`
sempre retornava `None`, então nenhuma linha carregava data e o filtro por
período era um no-op silencioso).

A análise é parametrizada pelo número de chamadas (`NUM_CALLS`, seção 1) —
quanto maior, mais completa e estatisticamente robusta fica a análise por KPI.
Todos os gráficos são interativos (Plotly): passe o mouse para ver detalhes,
dê zoom, exporte como PNG, etc.

**Observação:** a base de produção nunca traz um campo equivalente a
"qualidade do resumo" — a coluna `prod_qualidade_resumo` é sempre criada
(vazia, quando necessário) para manter a análise consistente com os 10 KPIs
oficiais de `config/kpis.yaml`.

Requer credenciais Azure válidas no ambiente (`az login` já realizado) e as
dependências opcionais instaladas: `pip install -e ".[azure,notebooks]"`.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.templates.default = "plotly_white"
PLOTLY_COLORWAY = px.colors.qualitative.Set2

from kpibench.data.azure_blob_loader import (
    load_interaction_jsons_from_azure,
    summarize_by_day,
    check_kpi_coverage,
    ALL_KPI_NAMES,
)


## 1. Parâmetros da análise completa e carga dos blobs

`NUM_CALLS` é o principal parâmetro desta análise: quanto maior, mais
chamadas são carregadas e mais robustas ficam as estatísticas por KPI (o
loader sempre eleva o valor para o mínimo de 10). Ajuste `START_DATE` /
`END_DATE` para o período desejado.


In [2]:
START_DATE = "2026-09-01"
END_DATE = "2026-10-28"
NUM_CALLS = 500  # defina o tamanho da análise completa (mínimo sempre elevado para 10)

df = load_interaction_jsons_from_azure(
    None,  # usa ACCOUNT_URL/CONTAINER_NAME/PATH_PREFIX fixos do módulo
    start_date=START_DATE,
    end_date=END_DATE,
    num_calls=NUM_CALLS,
)
print(f"Linhas carregadas: {len(df)}")
df.head()


[azure_blob_loader] Chamadas usadas na análise: 42
Linhas carregadas: 42


,metric_url,summary,dash_advins_fcr,dash_advins_odc,dash_advins_speed,dash_advins_assertiveness,dash_advins_flexibility,dash_advins_adaptability,dash_advins_empathy,dash_advins_resolution,...,complaint_prob_and_reasoning,prod_csat,prod_razao_csat,prod_solicitacao_resolvida,prod_solver_reasoning,prod_razao_ligacao,prod_first_call_resolution,prod_empatia,prod_clareza_informacoes,prod_odc
0,https://saatenadev.blob.core.windows.net/cs-br...,https://saatenadev.blob.core.windows.net/cs-br...,Não,Não,Não,Não,Não,Sim,Não,Não,...,NaN,7.0,"Edide recebeu informações importantes, mas a f...",nao,O problema não foi resolvido porque Edide não ...,Informações sobre título,nao,nao,sim,nao
1,https://saatenadev.blob.core.windows.net/cs-br...,https://saatenadev.blob.core.windows.net/cs-br...,Sim,Não,Não,Sim,Não,Sim,Não,Sim,...,NaN,6.0,Embora Gilberto tenha compreendido as informaç...,sim,O atendente explicou claramente as regras de r...,Cancelar título de capitalização,sim,nao,sim,nao
2,https://saatenadev.blob.core.windows.net/cs-br...,https://saatenadev.blob.core.windows.net/cs-br...,Não,Não,Não,Sim,Sim,Sim,Não,Sim,...,NaN,6.0,"Adivane pareceu confusa com a situação, mas Va...",sim,O cliente decidiu realizar o resgate do valor ...,Resgate de título capitalização.,nao,nao,sim,nao
3,https://saatenadev.blob.core.windows.net/cs-br...,https://saatenadev.blob.core.windows.net/cs-br...,Sim,Não,Não,Sim,Sim,Sim,Sim,Não,...,NaN,6.0,João pareceu satisfeito com a compreensão da a...,nao,O problema não foi resolvido porque João expre...,Verificar saldo título capitalização,sim,sim,sim,nao
4,https://saatenadev.blob.core.windows.net/cs-br...,https://saatenadev.blob.core.windows.net/cs-br...,Sim,Não,Sim,Sim,Não,Sim,Não,Sim,...,NaN,8.0,O cliente pareceu satisfeito ao receber as inf...,sim,O atendente forneceu informações claras sobre ...,Obter informações sobre crédito,sim,nao,sim,nao


In [3]:
# A base de produção nunca traz um campo equivalente a "qualidade do resumo"
# (ver check_kpi_coverage / KPI_DOMAIN_MISMATCH em azure_blob_loader.py) — a
# coluna prod_qualidade_resumo é sempre criada aqui, mesmo vazia (NaN), para
# que o restante da análise trate os 10 KPIs oficiais de forma uniforme.
if "prod_qualidade_resumo" not in df.columns:
    df["prod_qualidade_resumo"] = pd.NA
    print("Coluna 'prod_qualidade_resumo' ausente na base de produção — criada vazia (NaN).")
else:
    print("Coluna 'prod_qualidade_resumo' já presente na base.")


Coluna 'prod_qualidade_resumo' ausente na base de produção — criada vazia (NaN).


## 2.1 Cobertura dos KPIs do benchmark (config/kpis.yaml)

`load_interaction_jsons_from_azure` já deriva automaticamente, a partir de
`metrics_items`, uma coluna `prod_<kpi>` para cada um dos 10 KPIs oficiais do
projeto (quando existe um campo correspondente na base de produção).
`prod_qualidade_resumo` é a exceção conhecida — a base de produção nunca traz
esse campo, então a coluna é sempre criada vazia (célula anterior) para manter
os 10 KPIs consistentes nas tabelas e gráficos abaixo.

`check_kpi_coverage` informa:
- **ausentes** — KPIs sem nenhum campo de produção correspondente (ex.:
  `qualidade_resumo`, que não é calculado hoje em produção).
- **com divergência de domínio** — KPIs com dado disponível, mas em um formato
  diferente do esperado por `config/kpis.yaml` (ex.: campos binários Sim/Não
  onde o benchmark espera uma nota 1-5 ou uma taxonomia categórica fixa).

O gráfico interativo abaixo resume visualmente o status de cada KPI (OK /
divergência de domínio / ausente).


In [4]:
coverage = check_kpi_coverage(df)

prod_cols = [c for c in df.columns if c.startswith("prod_")]
print(f"\nColunas prod_<kpi> geradas ({len(prod_cols)} de {len(ALL_KPI_NAMES)} KPIs):")
display(df[["call_id", "data_referencia", *prod_cols]].head(10))

# Gráfico interativo de cobertura: para cada um dos 10 KPIs oficiais, mostra se
# há coluna prod_<kpi> (OK), se está ausente, ou se está presente mas com
# domínio diferente do esperado por config/kpis.yaml.
status_rows = [
    {
        "kpi": kpi,
        "status": (
            "Ausente" if kpi in coverage["ausentes"]
            else "Divergência de domínio" if kpi in coverage["com_divergencia_de_dominio"]
            else "OK"
        ),
    }
    for kpi in ALL_KPI_NAMES
]
coverage_df = pd.DataFrame(status_rows)
status_colors = {"OK": "#2f855a", "Divergência de domínio": "#dd6b20", "Ausente": "#c53030"}

fig_coverage = px.bar(
    coverage_df,
    x="kpi",
    y=[1] * len(coverage_df),
    color="status",
    color_discrete_map=status_colors,
    text="status",
    title="Cobertura dos 10 KPIs oficiais na base de produção",
    labels={"kpi": "KPI", "y": ""},
)
fig_coverage.update_traces(textposition="outside")
fig_coverage.update_yaxes(visible=False)
fig_coverage.update_layout(
    showlegend=True, xaxis_tickangle=-30, bargap=0.4, height=420,
    legend_title_text="Status",
)
fig_coverage.show()


Todos os KPIs de config/kpis.yaml têm uma coluna correspondente na base.

KPIs com dado de produção disponível, mas com domínio DIFERENTE do esperado:
  - solicitacao_resolvida: produção só distingue Sim/Não; kpis.yaml também espera 'parcial'
  - razao_ligacao: produção retorna texto livre (call_reason); kpis.yaml espera 1 dos 8 rótulos fixos
  - first_call_resolution: produção só distingue Sim/Não; kpis.yaml também espera 'indeterminado'
  - empatia: produção retorna Sim/Não; kpis.yaml espera nota ordinal 1-5
  - clareza_informacoes: produção retorna Sim/Não; kpis.yaml espera nota ordinal 1-5
  - odc: produção retorna Sim/Não; kpis.yaml espera taxonomia categórica de 7 rótulos

Colunas prod_<kpi> geradas (10 de 10 KPIs):


,call_id,data_referencia,prod_csat,prod_razao_csat,prod_solicitacao_resolvida,prod_solver_reasoning,prod_razao_ligacao,prod_first_call_resolution,prod_empatia,prod_clareza_informacoes,prod_odc,prod_qualidade_resumo
0,2b5d8efa-80b9-4b7e-b143-fb748bf94010,2026-09-15,7.0,"Edide recebeu informações importantes, mas a f...",nao,O problema não foi resolvido porque Edide não ...,Informações sobre título,nao,nao,sim,nao,<NA>
1,341fde30-df0e-4b8e-afee-7a992dac38ad,2026-09-15,6.0,Embora Gilberto tenha compreendido as informaç...,sim,O atendente explicou claramente as regras de r...,Cancelar título de capitalização,sim,nao,sim,nao,<NA>
2,86656231-6846-4072-a1ea-39554417d259,2026-09-15,6.0,"Adivane pareceu confusa com a situação, mas Va...",sim,O cliente decidiu realizar o resgate do valor ...,Resgate de título capitalização.,nao,nao,sim,nao,<NA>
3,8a2f014c-3a59-4b41-af3e-a595ba8d700b,2026-09-15,6.0,João pareceu satisfeito com a compreensão da a...,nao,O problema não foi resolvido porque João expre...,Verificar saldo título capitalização,sim,sim,sim,nao,<NA>
4,a6c0a3d6-ff1f-4c48-aee8-38a2ab827381,2026-09-15,8.0,O cliente pareceu satisfeito ao receber as inf...,sim,O atendente forneceu informações claras sobre ...,Obter informações sobre crédito,sim,nao,sim,nao,<NA>
5,50786027-e966-488a-aea9-819d9dde6d0e,2026-09-16,9.0,Maria Luísa parece satisfeita com o atendiment...,sim,"O problema foi resolvido, pois Maria Luísa con...",Contratação de título de capitalização.,sim,nao,sim,nao,<NA>
6,50a465fe-d1a2-4b6d-b9a1-47584ea44a1e,2026-09-16,6.0,A Maria pareceu estar neutra quanto à situação...,nao,"O problema do cliente não foi resolvido, pois ...",Pagamento e saldo resgate.,nao,nao,sim,nao,<NA>
7,82ade3c3-4261-4b40-ae3d-cbe04f3695c9,2026-09-16,4.0,Renato ficou insatisfeito com a necessidade de...,nao,O problema de Renato não foi resolvido porque ...,Resgate de título.,nao,nao,sim,nao,<NA>
8,aef1a47f-a8cb-431a-bbf7-873ebe4d90a5,2026-09-16,6.0,Apesar de Ana ter fornecido as informações nec...,nao,O problema não foi resolvido porque Jefferson ...,Informações sobre resgate.,nao,sim,sim,nao,<NA>
9,cf0b8547-601d-4c75-8a9f-5e0932f99f58,2026-09-16,8.0,Tereza ficou satisfeita com a informação sobre...,sim,"O problema foi resolvido, pois Tereza recebeu ...",Confirmação de resgate.,sim,sim,sim,nao,<NA>


## 2. Checagem de qualidade: cada linha tem `call_id` e `data_referencia`?

Antes da correção, `data_referencia` nem existia como coluna. Se os percentuais
abaixo não forem 0%, há blobs cujo caminho não bate com o padrão `date=YYYYMMDD`
esperado — vale inspecionar `blob_name` desses casos.

In [5]:
missing_call_id = df["call_id"].isna().mean() * 100 if "call_id" in df else None
missing_date = df["data_referencia"].isna().mean() * 100 if "data_referencia" in df else None
print(f"% sem call_id: {missing_call_id:.2f}%")
print(f"% sem data_referencia: {missing_date:.2f}%")

if missing_date and missing_date > 0:
    display(df.loc[df["data_referencia"].isna(), ["call_id", "blob_name"]].head(20))

fig_quality = px.bar(
    pd.DataFrame(
        {
            "campo": ["call_id", "data_referencia"],
            "percentual_ausente": [missing_call_id or 0, missing_date or 0],
        }
    ),
    x="campo",
    y="percentual_ausente",
    text="percentual_ausente",
    color="campo",
    color_discrete_sequence=PLOTLY_COLORWAY,
    title="Qualidade dos dados: % de valores ausentes",
    labels={"percentual_ausente": "% ausente", "campo": "Campo"},
)
fig_quality.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig_quality.update_layout(
    showlegend=False,
    yaxis_range=[0, max(5.0, missing_call_id or 0, missing_date or 0) * 1.3],
    height=380,
)
fig_quality.show()


% sem call_id: 0.00%
% sem data_referencia: 0.00%


## 3. Agrupamento por dia

`summarize_by_day` conta chamadas por dia e calcula a média das colunas
numéricas (ex.: CSAT, tokens). Use esta tabela/gráfico para validar visualmente
se o volume diário é coerente com o esperado (sem dias "vazios" ou picos
anômalos causados por bug de parsing).

In [6]:
daily = summarize_by_day(df)
daily

,dia,total_chamadas,prod_csat
0,2026-09-15,5,6.600000
1,2026-09-16,5,6.600000
2,2026-09-17,10,5.400000
3,2026-09-18,10,5.100000
4,2026-09-19,5,5.800000
5,2026-09-20,7,5.714286


In [7]:
avg_csat_col = "prod_csat" if "prod_csat" in daily.columns else None

fig_daily = go.Figure()
fig_daily.add_bar(
    x=daily["dia"].astype(str),
    y=daily["total_chamadas"],
    name="Total de chamadas",
    marker_color="#2b6cb0",
)

if avg_csat_col:
    fig_daily.add_trace(
        go.Scatter(
            x=daily["dia"].astype(str),
            y=daily[avg_csat_col],
            name="CSAT médio",
            mode="lines+markers",
            yaxis="y2",
            line=dict(color="#dd6b20", width=3),
        )
    )
    fig_daily.update_layout(yaxis2=dict(title="CSAT médio", overlaying="y", side="right", showgrid=False))

fig_daily.update_layout(
    title="Chamadas por dia" + (" e CSAT médio" if avg_csat_col else ""),
    xaxis_title="Dia",
    yaxis_title="Total de chamadas",
    template="plotly_white",
    height=450,
    xaxis_tickangle=-60,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_daily.show()


## 3.1 Distribuição dos KPIs de produção disponíveis

Visão geral interativa de todos os `prod_<kpi>` presentes na base: histograma
para KPIs numéricos (ex.: `prod_csat`) e contagem de categorias para KPIs
categóricos (ex.: Sim/Não). `prod_qualidade_resumo` aparece vazio, pois é
sempre criado sem dado real (ver seção 1).


In [8]:
present_prod_cols = [c for c in prod_cols if c in df.columns]
n_cols = 3
n_rows = -(-len(present_prod_cols) // n_cols)  # ceil division

fig_dist = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[c.replace("prod_", "") for c in present_prod_cols],
)

for i, col in enumerate(present_prod_cols):
    row, c = divmod(i, n_cols)
    row, c = row + 1, c + 1
    series = df[col].dropna()
    color = PLOTLY_COLORWAY[i % len(PLOTLY_COLORWAY)]
    if series.empty:
        continue
    if pd.api.types.is_numeric_dtype(series):
        fig_dist.add_trace(
            go.Histogram(x=series, marker_color=color, name=col, showlegend=False),
            row=row, col=c,
        )
    else:
        counts = series.value_counts()
        fig_dist.add_trace(
            go.Bar(x=counts.index.astype(str), y=counts.values, marker_color=color, name=col, showlegend=False),
            row=row, col=c,
        )

fig_dist.update_layout(
    height=320 * n_rows,
    title_text="Distribuição dos KPIs de produção disponíveis",
    template="plotly_white",
)
fig_dist.show()


## 4. Checagem cruzada: data da partição vs. `timestamp` interno do payload

Alguns blobs trazem um campo `timestamp` dentro do JSON (data/hora real do
processamento). Se existir, compare com `data_referencia` (data da pasta do
blob) para confirmar que o particionamento bate com o conteúdo real — uma
divergência grande indica atraso de pipeline ou blobs arquivados na pasta
errada.

In [9]:
if "timestamp" in df.columns:
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    diff_days = (df["data_referencia"] - ts.dt.normalize()).dt.days
    mismatch = diff_days[diff_days.abs() > 0]
    print(f"Registros com timestamp em dia diferente da partição: {len(mismatch)} de {len(df)}")
    if len(mismatch):
        display(df.loc[mismatch.index, ["call_id", "data_referencia", "timestamp"]].head(20))

    diff_days_valid = diff_days.dropna()
    if not diff_days_valid.empty:
        fig_ts = px.histogram(
            diff_days_valid,
            nbins=min(30, max(1, diff_days_valid.nunique())),
            title="Diferença (dias) entre partição do blob e timestamp interno",
            labels={"value": "Diferença em dias"},
            color_discrete_sequence=["#2b6cb0"],
        )
        fig_ts.update_layout(showlegend=False, template="plotly_white", height=380, yaxis_title="Chamadas")
        fig_ts.show()
else:
    print("Coluna 'timestamp' não encontrada no payload — checagem pulada.")


Registros com timestamp em dia diferente da partição: 42 de 42


,call_id,data_referencia,timestamp
0,2b5d8efa-80b9-4b7e-b143-fb748bf94010,2026-09-15,2026-09-24 12:29:14.759618
1,341fde30-df0e-4b8e-afee-7a992dac38ad,2026-09-15,2026-09-24 12:29:15.150482
2,86656231-6846-4072-a1ea-39554417d259,2026-09-15,2026-09-24 12:29:14.892449
3,8a2f014c-3a59-4b41-af3e-a595ba8d700b,2026-09-15,2026-09-24 12:29:15.021865
4,a6c0a3d6-ff1f-4c48-aee8-38a2ab827381,2026-09-15,2026-09-24 12:29:14.574217
5,50786027-e966-488a-aea9-819d9dde6d0e,2026-09-16,2026-09-23 20:58:58.534735
6,50a465fe-d1a2-4b6d-b9a1-47584ea44a1e,2026-09-16,2026-09-23 20:58:58.003240
7,82ade3c3-4261-4b40-ae3d-cbe04f3695c9,2026-09-16,2026-09-23 20:58:58.367325
8,aef1a47f-a8cb-431a-bbf7-873ebe4d90a5,2026-09-16,2026-09-23 20:58:58.724540
9,cf0b8547-601d-4c75-8a9f-5e0932f99f58,2026-09-16,2026-09-23 20:58:58.207996
